# Fine-tune a tiny receipt reader (AgentLab Step 4.3)

This notebook teaches **Gemma 3 270M** one job: read a receipt's text and answer with its fields as JSON (vendor, date, subtotal, GST, other tax, tip, total).

- **Data:** 2,000 **fictional** receipts made by [`make_training_data.py`](https://github.com/GurminderKaur/AgentLab/blob/main/07-fine-tuning/data/make_training_data.py), plus 200 held out for checking. Nothing real is uploaded.
- **Method:** LoRA with [Unsloth](https://github.com/unslothai/unsloth): only a small add-on is trained; the original model stays frozen.
- **Output:** one `receipt-reader-Q8_0.gguf` file (~0.3 GB) to run in Ollama on my laptop.

**Before you start:** *Runtime → Change runtime type → **T4 GPU** → Save.* Then run the cells top to bottom (▶ on each, or *Runtime → Run all*). It takes about 15–25 minutes in total.

## 1. Install the tools (about 2–3 minutes)

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 2. Get the fictional training data from GitHub

In [ ]:
!wget -q -O train.jsonl https://raw.githubusercontent.com/GurminderKaur/AgentLab/main/07-fine-tuning/data/train.jsonl
!wget -q -O heldout.jsonl https://raw.githubusercontent.com/GurminderKaur/AgentLab/main/07-fine-tuning/data/heldout.jsonl
import json
heldout = [json.loads(l) for l in open("heldout.jsonl")]
print(sum(1 for _ in open("train.jsonl")), "training receipts,", len(heldout), "held out")
print(heldout[0]["messages"][0]["content"][-400:])
print("→", heldout[0]["messages"][1]["content"])

## 3. Load Gemma 3 270M and add a LoRA adapter

Only the adapter (a small set of new numbers) will be trained.

In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = 2048,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)
model = FastModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma3")

## 4. A scorer: how many fields does the model get right?

Same rules as ReceiptBench on my laptop: amounts to the cent, date exact, vendor by name.

In [ ]:
import re
FIELDS = ["vendor", "date", "subtotal", "gst", "other_tax", "tip", "total"]

def ask(user_text):
    text = tokenizer.apply_chat_template([{"role": "user", "content": user_text}], tokenize = False, add_generation_prompt = True).removeprefix('<bos>')
    inputs = tokenizer(text, return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = 160, do_sample = False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)

def norm(s): return re.sub(r"[^a-z0-9]", "", str(s).lower())
def num(v):
    try: return float(re.sub(r"[^0-9.\-]", "", str(v)) or 0)
    except ValueError: return 0.0

def score(examples, show = 2):
    hits = {f: 0 for f in FIELDS}
    for i, ex in enumerate(examples):
        want = json.loads(ex["messages"][1]["content"])
        reply = ask(ex["messages"][0]["content"])
        m = re.search(r"\{.*\}", reply, re.S)
        try: got = json.loads(m.group(0)) if m else {}
        except json.JSONDecodeError: got = {}
        ok = {
            "vendor": len(norm(got.get("vendor", ""))) > 4 and (norm(got.get("vendor", "")) in norm(want["vendor"]) or norm(want["vendor"]) in norm(got.get("vendor", ""))),
            "date": got.get("date") == want["date"],
            **{f: abs(num(got.get(f, 0)) - want[f]) < 0.005 for f in ["subtotal", "gst", "other_tax", "tip", "total"]},
        }
        for f in FIELDS: hits[f] += ok[f]
        if i < show: print("model said:", reply.strip()[:200], "\nright was:", ex["messages"][1]["content"], "\n")
    n = len(examples)
    print(", ".join(f"{f} {hits[f]}/{n}" for f in FIELDS), f"· all fields right on {sum(hits.values())}/{n * len(FIELDS)}")
    return hits

## 5. Before training: score 50 held-out receipts

Expect poor results: in ReceiptBench the untrained model copied the example shape (`"..."`, `"YYYY-MM-DD"`, zeros).

In [ ]:
before = score(heldout[:50])

## 6. Train (about 5–10 minutes on a T4)

The loss (how wrong the model's answers are) should fall quickly and then level off. Only the **answers** are scored during training, not the questions (`train_on_responses_only`).

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files = "train.jsonl", split = "train")
def formatting(examples):
    return {"text": [tokenizer.apply_chat_template(m, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for m in examples["messages"]]}
dataset = dataset.map(formatting, batched = True)

from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 1,
        warmup_steps = 10,
        num_train_epochs = 2,          # two passes over the 2,000 receipts
        learning_rate = 1e-4,
        logging_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)
stats = trainer.train()
print(f"Trained in {stats.metrics['train_runtime'] / 60:.1f} minutes")

## 7. After training: score all 200 held-out receipts

These were never trained on. A big jump here means the model learned the rules, not just the examples.

In [ ]:
after = score(heldout, show = 3)

## 8. Export to one GGUF file for Ollama, and download it (~0.3 GB)

Your browser will ask where to save `receipt-reader-Q8_0.gguf`. Save it into your `AgentLab\\07-fine-tuning` folder, next to the `Modelfile`.

In [ ]:
import glob, shutil
model.save_pretrained_gguf("receipt_reader", tokenizer, quantization_method = "Q8_0")
ggufs = sorted(glob.glob("receipt_reader*/*.gguf") + glob.glob("receipt_reader*.gguf") + glob.glob("*.gguf"), key = os.path.getsize, reverse = True)
print("GGUF files found:", ggufs)
shutil.copy(ggufs[0], "receipt-reader-Q8_0.gguf")
print(f"receipt-reader-Q8_0.gguf: {os.path.getsize('receipt-reader-Q8_0.gguf') / 1e6:.0f} MB")
from google.colab import files
files.download("receipt-reader-Q8_0.gguf")

## 9. Next, on the laptop (Step 4.4)

In PowerShell, in your AgentLab folder:

```powershell
git pull
cd 07-fine-tuning
ollama create receipt-reader -f Modelfile
cd ReceiptBench
dotnet run -- extract receipt-reader
```

Then compare with the "before" numbers (11/42) and Gemma 4 (42/42).